In [ ]:
!pip install neo4j

from neo4j import GraphDatabase
import pandas as pd

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 6.9 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
URI = "neo4j+s://f721593f.databases.neo4j.io"
AUTH = ("neo4j", "fHJJUbw9LfSVMqxTcBpm91_U6divfJfHWRLtXNqi1gk")
files_config = {
    'out_train': '/content/drive/My Drive/INŻYNIERKA/neo4j_train_reviews.csv',
    'out_test': '/content/drive/My Drive/INŻYNIERKA/neo4j_test_reviews.csv',
    'out_prod': '/content/drive/My Drive/INŻYNIERKA/neo4j_products.csv'
}

In [ ]:
class GraphImporter:
  def __init__(self, uri, auth):
      self.driver = GraphDatabase.driver(uri, auth=auth)

  def close(self):
      self.driver.close()

  def clean_database(self):
      print("Cleaning database...")
      with self.driver.session() as session:
          session.run("MATCH (n) DETACH DELETE n") # Nie usuwamy constraints bo z nimi i tak nie bedzie problemu
      print("Database wiped.")

  def create_constraints(self):
      queries = [
          "CREATE CONSTRAINT user_id_unique IF NOT EXISTS FOR (u:User) REQUIRE u.user_id IS UNIQUE",
          "CREATE CONSTRAINT product_id_unique IF NOT EXISTS FOR (p:Product) REQUIRE p.parent_asin IS UNIQUE",
          "CREATE CONSTRAINT review_id_unique IF NOT EXISTS FOR (r:Review) REQUIRE (r.user_id,r.parent_asin) IS UNIQUE",
      ]

      with self.driver.session() as session:
          for q in queries:
              session.run(q)
          print("Constraints created.")

  def import_products(self, csv_path, batch_size = 5000):
    # zachowujemy bezpieczenstwo: MERGE gdyby cokolwiek sie powtorzylo
    # unwind tworzy petle po stronie bazy, wiec obliczenia w bazie nie w pythonie
    query = """
        UNWIND $batch AS row
        MERGE (p:Product {parent_asin: row.parent_asin})
        ON CREATE SET p.title = row.title,
          p.average_rating = toFloat(row.average_rating),
          p.rating_number = toInteger(row.rating_number),
          p.price = CASE WHEN isNaN(toFloat(row.price)) THEN null ELSE toFloat(row.price) END,
          p.store = row.store,
          p.description = split(row.description, '|'),
          p.features = split(row.features, '|'),
          p.images = split(row.images, '|')
    """

    print("Importing products...")
    text_cols = ['title', 'store', 'description', 'features', 'images']
    #numeric_cols = ['average_rating','rating_number','price']
    count = 0
    with self.driver.session() as session:
      for chunk in pd.read_csv(csv_path, chunksize=batch_size):
        # Pandas zwraca teraz iterator, a nie wielki DataFrame
        chunk[text_cols] = chunk[text_cols].fillna('')  # zamieniamy tylko pola tekstowe, numeryczne zostawimy jako nulle w razie braku
        # chunk = chunk.where(pd.notnull(chunk), None) #notna - zwraca true/false tabele czy wartosci sa nie NA: czyli numpy.NaN oraz None pythonowe sa mapowane do false
        rows = chunk.to_dict(orient='records') # dict with each row as dict with column_name:value pairs
        session.run(query, batch=rows)
        count += len(rows)
        print(f"Processed {count}...")
    print("Done importing products.")


  def import_reviews(self, csv_path, split_type, batch_size = 5000):
    # znajdz istniejacy produkt, znajdujemy uzytkownika o podanym id lub tworzymy go (czyli tworzymy albo nic sie nie dzieje) i dla tych nodow ktore teraz juz na pewno sa w bazie albo sprawdzono ze produkt o ktorym mowa w recenzji jest w bazie, tworzymy node review
    query = """
    UNWIND $batch AS row
    // znajdz produkt i uzytkownika, utworz uzytkownika jezeli trzeba
    MATCH (p:Product {parent_asin: row.parent_asin})
    MERGE (u:User {user_id: row.user_id})

    // tworzymy albo laczymy z recenzja
    MERGE (r:Review {
      user_id: row.user_id,
      parent_asin: row.parent_asin
    })
    ON CREATE SET
      r.rating = toFloat(row.rating),
      r.timestamp = toInteger(row.timestamp),
      r.title = row.title,
      r.text = row.text,
      r.helpful_vote = toInteger(row.helpful_vote),
      r.verified_purchase = toBoolean(row.verified_purchase),
      r.images = split(row.images, '|'),
      r.split = $split_type

    CREATE (u)-[:WROTE]->(r)
    CREATE (r)-[:ABOUT]->(p)

    MERGE (u)-[rated:RATED]->(p)
    SET rated.rating = toFloat(row.rating),
        rated.split = $split_type
    """
    print(f"Importing {split_type} reviews...")
    text_cols = ['title', 'text', 'images']
    count = 0
    with self.driver.session() as session:
      for chunk in pd.read_csv(csv_path, chunksize=batch_size):
        chunk[text_cols] = chunk[text_cols].fillna('')
        rows = chunk.to_dict('records')
        session.run(query, batch=rows, split_type=split_type)
        count += len(rows)
        print(f"Imported {count} reviews...")
    print(f"Done importing {split_type} reviews.")



In [ ]:
importer = GraphImporter(URI, AUTH)

try:
  importer.clean_database() # Added parentheses
  importer.create_constraints()
  importer.import_products(files_config['out_prod'])
  importer.import_reviews(files_config['out_train'], 'train')
  importer.import_reviews(files_config['out_test'], 'test')
finally:
  importer.close()
  print("Database import finished.")

Cleaning database...
Database wiped.
Constraints created.
Importing products...
Processed 356...
Done importing products.
Importing train reviews...
Imported 2282 reviews...
Done importing train reviews.
Importing test reviews...
Imported 253 reviews...
Done importing test reviews.
Database import finished.
